<div align="center">

# Universidad de Sevilla  
## Grado en Ingeniería del Software  
### Escuela Técnica Superior de Ingeniería Informática  
### Inteligencia Artificial (IA) – Curso 2024/2025  

<img src="../../img/portada.png" width="300"/>

---

#  Ensamble Secuencial de Modelos Predictivos  
### Primera Convocatoria – Junio 2025  

**Profesor:** Juan Galán Páez  
**Autores:**  
Carlos Martín de Prado Barragán  
Marco Padilla Gómez  

**Fecha de entrega:** 9 de Junio de 2025  

</div>

## Introducción

En este notebook se realiza una exploración exhaustiva de los hiperparámetros del meta-modelo secuencial aplicado al conjunto de datos sobre los precios de la vivienda.

Mediante validación cruzada con 5 particiones, se evalúan diferentes combinaciones de parámetros clave como el número de estimadores, la tasa de aprendizaje y el numero de ejemplos.

Además del coeficiente de determinación R², se incluyen métricas complementarias como el Error Absoluto Medio (MAE) y la Raíz del Error Cuadrático Medio (RMSE) para proporcionar una evaluación más completa del rendimiento.

El objetivo es identificar las configuraciones que maximicen la capacidad predictiva y la robustez del modelo, fundamentando así las decisiones de diseño y optimización.

## 1. Importación de librerías y configuración del entorno

- Se añade la ruta `../../src` para importar el módulo personalizado `sequential_ensemble.py` que contiene la implementación del meta-modelo secuencial.
- Se importan las librerías necesarias para el procesamiento de datos (`numpy`, `pandas`), modelado (`LinearRegression`), preprocesamiento (`StandardScaler`, `OneHotEncoder`), validación cruzada (`cross_validate`, `KFold`), y métricas de evaluación (`mean_absolute_error`, `mean_squared_error`).
- Finalmente, se importa la clase `SequentialEnsembleRegressor` para utilizar el meta-modelo en los experimentos.

In [1]:
# Importación de librerías necesarias
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import cross_validate, KFold
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.compose import ColumnTransformer

# Importar el meta-modelo desde nuestro módulo
import sys
sys.path.append('../../src')

from sequential_ensemble import SequentialEnsembleRegressor

## 2. Carga y preprocesamiento del dataset

- Se carga el conjunto de datos `house_prices.csv` desde la carpeta `data`.
- Se separan las variables predictoras (`X`) y la variable objetivo (`y`), que corresponde a la escala `SalePrice`.
- Se aplica estandarización a las variables predictoras utilizando `StandardScaler` para normalizar su escala y facilitar el entrenamiento del modelo y `OneHotEncoder` para codificar las variables categóricas.
- Se combinan ambas transformaciones con un `ColumnTransformer`.

In [2]:
# Cargar el dataset de Parkinson
df = pd.read_csv("../../data/house_prices.csv")

# Separar variables predictoras y variable objetivo
X = df.drop(columns=["SalePrice"])
y = df["SalePrice"]

# Detectar variables categóricas (tipo objeto) y numéricas
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()

# Pipeline de preprocesamiento
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
])

# Aplicar el preprocesamiento al conjunto de datos
X_processed = preprocessor.fit_transform(X)

## 3. Definición del espacio de búsqueda de hiperparámetros

- Se establecen los rangos de valores para los hiperparámetros clave a explorar:
  - `n_estimators`: número de modelos base en el ensamblador, con valores 30, 50 y 200.
  - `learning_rate`: tasa de aprendizaje que modula la contribución de cada modelo, con valores 0.05, 0.2 y 0.5.
  - `sample_size`: porcentaje del conjunto de entrenamiento en cada iteración.
- Se inicializa una lista vacía `results` para almacenar los resultados de cada combinación probada durante la experimentación.

In [3]:
# Definimos rangos para los hiperparámetros más relevantes
n_estimators_options = [30, 50, 200]
learning_rate_options = [0.05, 0.2, 0.5]
sample_size_options = [0.75, 0.9, 1]  

# Para guardar resultados
results = []

## 4. Ejecución de experimentos con validación cruzada

- Se configura la validación cruzada con 5 particiones (`KFold`), mezclando aleatoriamente los datos para mayor representatividad.
- Se realiza un bucle anidado para probar todas las combinaciones posibles de los hiperparámetros definidos: número de estimadores, tasa de aprendizaje y porcentaje del conjunto de entreamiento.
- Para cada configuración, se crea una instancia del meta-modelo secuencial con esos parámetros.
- Se ejecuta la validación cruzada calculando múltiples métricas:  
  - Coeficiente de determinación R²,  
  - Error Absoluto Medio (MAE),  
  - Raíz del Error Cuadrático Medio (RMSE).
- Los resultados (medias y desviaciones estándar) se almacenan en la lista `results` para su posterior análisis.
- Se imprime un resumen en cada iteración para seguimiento en tiempo real de los experimentos.

In [4]:
# Configurar validación cruzada
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Métricas a evaluar como diccionario para evitar NaN
scoring = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error',
    'mse': 'neg_mean_squared_error'
}

# Bucle para probar combinaciones con validación cruzada
for sample_size in sample_size_options:
    for n_estimators in n_estimators_options:
        for lr in learning_rate_options:
            modelo = SequentialEnsembleRegressor(
                base_estimator=LinearRegression,
                n_estimators=n_estimators,
                lr=lr,
                sample_size=sample_size,
                est_params={},
                random_state=42
            )
            scores = cross_validate(modelo, X_processed, y, cv=kf, scoring=scoring)

            mean_r2 = np.mean(scores['test_r2'])
            mean_mae = -np.mean(scores['test_mae'])
            mean_rmse = np.sqrt(-np.mean(scores['test_mse']))
            std_r2 = np.std(scores['test_r2'])

            results.append({
                "sample_size": sample_size,
                "n_estimators": n_estimators,
                "learning_rate": lr,
                "mean_r2": mean_r2,
                "std_r2": std_r2,
                "mean_mae": mean_mae,
                "mean_rmse": mean_rmse
            })

            print(f"Evaluado sample_size={sample_size}, n_estimators={n_estimators}, "
                    f"lr={lr} -> R2 mean: {mean_r2:.4f}")

Evaluado sample_size=0.75, n_estimators=30, lr=0.05 -> R2 mean: 0.4866
Evaluado sample_size=0.75, n_estimators=30, lr=0.2 -> R2 mean: 0.7125
Evaluado sample_size=0.75, n_estimators=30, lr=0.5 -> R2 mean: 0.6949
Evaluado sample_size=0.75, n_estimators=50, lr=0.05 -> R2 mean: 0.7073
Evaluado sample_size=0.75, n_estimators=50, lr=0.2 -> R2 mean: 0.7211
Evaluado sample_size=0.75, n_estimators=50, lr=0.5 -> R2 mean: 0.7232
Evaluado sample_size=0.75, n_estimators=200, lr=0.05 -> R2 mean: 0.7205
Evaluado sample_size=0.75, n_estimators=200, lr=0.2 -> R2 mean: 0.7152
Evaluado sample_size=0.75, n_estimators=200, lr=0.5 -> R2 mean: 0.6985
Evaluado sample_size=0.9, n_estimators=30, lr=0.05 -> R2 mean: 0.4933
Evaluado sample_size=0.9, n_estimators=30, lr=0.2 -> R2 mean: 0.7279
Evaluado sample_size=0.9, n_estimators=30, lr=0.5 -> R2 mean: 0.7256
Evaluado sample_size=0.9, n_estimators=50, lr=0.05 -> R2 mean: 0.7126
Evaluado sample_size=0.9, n_estimators=50, lr=0.2 -> R2 mean: 0.7238
Evaluado sample_s

## 5. Análisis y visualización de resultados

- Se convierte la lista de resultados en un DataFrame de Pandas para facilitar su manipulación.
- Se ordenan las configuraciones probadas según el valor medio del coeficiente de determinación R², de mayor a menor, para identificar rápidamente las mejores combinaciones.
- Se reinicia el índice del DataFrame para mantener la consistencia tras el ordenamiento.
- Finalmente, se muestra la tabla con todas las configuraciones evaluadas y sus respectivos resultados, facilitando el análisis comparativo.

In [6]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="mean_r2", ascending=False)
results_df.reset_index(drop=True, inplace=True)
results_df

,sample_size,n_estimators,learning_rate,mean_r2,std_r2,mean_mae,mean_rmse
0,0.90,200,0.50,0.731659,0.127070,26265.687591,40030.715759
1,0.90,200,0.20,0.729240,0.122457,26486.528247,40324.669432
2,0.90,30,0.20,0.727939,0.124566,26556.657011,40305.135347
3,0.90,200,0.05,0.726811,0.122353,26609.984744,40523.855804
4,1.00,30,0.20,0.726321,0.124215,26611.430848,40509.573078
5,1.00,200,0.05,0.725954,0.125076,26633.367613,40527.483387
6,1.00,50,0.20,0.725948,0.125091,26633.766943,40527.804371
7,1.00,30,0.50,0.725943,0.125102,26634.041197,40528.025046
8,1.00,50,0.50,0.725943,0.125102,26634.041215,40528.025060
9,1.00,200,0.50,0.725943,0.125102,26634.041215,40528.025060


In [7]:
# Comparación del mejor ensamblador con el modelo base individual (DecisionTreeRegressor)
from sklearn.model_selection import cross_val_score

# Extraer la mejor configuración del ensamblador
mejor_config = results_df.iloc[0]

# Crear y evaluar el modelo base individual
modelo_base = LinearRegression()
r2_base = np.mean(cross_val_score(modelo_base, X_processed, y, cv=kf, scoring='r2'))

# Mostrar comparación
print(f"R² del mejor ensamblador (validación cruzada): {mejor_config['mean_r2']:.4f}")
print(f"R² del modelo base individual: {r2_base:.4f}")

R² del mejor ensamblador (validación cruzada): 0.7317
R² del modelo base individual: 0.7259


## 6. Interpretación de resultados y conclusiones

Los experimentos realizados evidencian que el rendimiento del meta-modelo secuencial entrenado con `LinearRegression` está condicionado por la configuración de sus hiperparámetros, aunque las diferencias en desempeño son moderadas debido a las limitaciones del estimador lineal como modelo base.

- Cuando se emplean tasas de aprendizaje bajas (`learning_rate=0.05`) combinadas con un número reducido de estimadores (`n_estimators=30` o `50`), el rendimiento del modelo cae de forma significativa, llegando a obtener valores de R² por debajo de **0.5**, lo que indica una pobre capacidad de ajuste.

- El rendimiento mejora al **incrementar el número de estimadores** (`n_estimators=200`) y combinarlo con tasas de aprendizaje medias (`lr=0.2` o `0.5`), alcanzando valores de R² en torno a **0.73**. La mejor configuración fue con `sample_size=0.90`, `n_estimators=200`, `learning_rate=0.50`, que logró un R² promedio de **0.7317**.

- La métrica **mean_mae** (error absoluto medio) osciló entre **26,200 y 42,000**, reflejando una clara diferencia de precisión entre las configuraciones óptimas y las menos adecuadas.

- El parámetro `sample_size` también influye: valores intermedios y altos (como **0.90 o 1.00**) suelen estar presentes en las configuraciones con mejor rendimiento. Estos valores permiten a cada modelo del ensamble aprender de una proporción más representativa del conjunto de entrenamiento.

- La **desviación estándar del R²** se mantuvo en general entre **0.11 y 0.13**, indicando un comportamiento razonablemente estable entre los distintos folds, salvo en configuraciones especialmente malas.

- Aunque la regresión lineal limita la capacidad del meta-modelo para capturar relaciones no lineales, los resultados demuestran que el ensamble secuencial **puede mejorar sistemáticamente el rendimiento respecto al modelo base**, siempre que se utilice una configuración de hiperparámetros adecuada.